# 04 · From one leaflet to a whole branch: LLS-SevEst 2.0 (2025)

The first model measured one leaflet per image. For
2.0 the leaflets of a whole lateral branch are scanned together (six branches, scanned on 24/04/2025), which brought
two new problems:

1. **Segmentation.** The original model marked veins and pale parts of the blade as lesion in some leaflets.
   I tried many combinations of extra pixel features (section 1).
2. **Counting.** To account for defoliation the tool has to count the leaflets and compare with the expected number
   (4 per node). Section 3 is the search for the parameters of the counter.

Cleaned-up version of the Colab notebook of May–June 2025. The scans are **not** included (available on reasonable
request): put them in `data/scans/` as `IMG_20250424_000<n>.tif`. Sections 1 and 2 need the scans. Section 3's
analysis runs without them, from the real counts in `data/leaflet_counts_*.csv`. Saved **without outputs**.

> The tables in `data/` that this notebook reads (`*.csv`) are **not included in the repository**; they are available on reasonable request, together with the scans.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.feature import local_binary_pattern
from sklearn import preprocessing
from sklearn.cluster import KMeans

SCANS_DIR = Path("../../data/scans")      # not included: IMG_20250424_000<n>.tif
DATA_DIR = Path("../../data")
SCAN_1 = SCANS_DIR / "IMG_20250424_0001.tif"


def read_rgb(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## 1. Segmentation: which pixel features?

The 2024 model used 6 features per pixel (R, G, B and the three differences). For branches I built one function that can
compute every feature I tried, and ran K-means (3 groups, standardised features, `random_state=2206`) on each
combination. **I judged the result by looking at the segmentation** (veins and pale areas marked as lesion or not);
there was no numerical comparison, and none of the 2.0 results was compared with Pliman or any other reference.

Several of the ideas (gradient, brightness, position, distance to the centre, local binary patterns, and HSV colour
as a general direction, which was already on my to-do list) came from an AI assistant (DeepSeek, 14–15/05/2025) that I
asked for more pixel features; I then tested them one by one. The feature set that looked best is the 12-feature one
(`final_12`), the one in the package.

`FEATURE_SETS` lists the trials in the order I ran them: the features, the severity printed by the original notebook for
scan 1 (`None` where it only appeared in a plot title) and the group that trial counted as lesion. Read the note in
section 2 before comparing the numbers.

In [ ]:
def build_features(img_rgb):
    """Every per-pixel feature tried in 2.0. Each trial used a subset (see FEATURE_SETS)."""
    blur = cv2.medianBlur(img_rgb, 11)
    rows, cols = blur.shape[:2]
    r, c = np.indices((rows, cols))
    px = blur.astype(int)
    f = pd.DataFrame({"row": r.ravel(), "col": c.ravel(),
                      "red": px[..., 0].ravel(), "green": px[..., 1].ravel(), "blue": px[..., 2].ravel()})
    f["diff_r_g"] = f["red"] - f["green"]
    f["diff_r_b"] = f["red"] - f["blue"]
    f["diff_b_g"] = f["blue"] - f["green"]

    hsv = cv2.cvtColor(blur, cv2.COLOR_RGB2HSV)
    f["hue"], f["saturation"], f["value"] = (hsv[..., k].ravel() for k in range(3))
    f["sat_norm"], f["val_norm"] = f["saturation"] / 255.0, f["value"] / 255.0

    gray = cv2.cvtColor(blur, cv2.COLOR_RGB2GRAY)
    gx, gy = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5), cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5)
    f["gradient"] = np.sqrt(gx ** 2 + gy ** 2).ravel()
    f["brightness"] = 0.299 * f["red"] + 0.587 * f["green"] + 0.114 * f["blue"]

    f["rel_row"], f["rel_col"] = f["row"] / rows, f["col"] / cols
    f["dist_to_center"] = np.sqrt((f["row"] - rows / 2) ** 2 + (f["col"] - cols / 2) ** 2)
    total = f[["red", "green", "blue"]].sum(axis=1)
    f["norm_r"], f["norm_g"], f["norm_b"] = f["red"] / total, f["green"] / total, f["blue"] / total
    f["lbp"] = local_binary_pattern(gray, 8 * 3, 3, method="uniform").ravel()

    # Used only in the "texture + morphology" trial (that trial used a Sobel kernel of 3, not 5)
    f["gabor"] = cv2.filter2D(gray, cv2.CV_64F, cv2.getGaborKernel((15, 15), 3, np.pi / 4, 10.0, 0.5)).ravel()
    adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    closed = cv2.morphologyEx(cv2.morphologyEx(adaptive, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=2),
                              cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8), iterations=2)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    perimeter = cv2.arcLength(contours[0].astype(np.int32), True) if contours else 0.0
    area = cv2.moments(closed)["m00"]
    # These three are one number per image, repeated for every pixel: after standardising they carry no information.
    f["perimeter"], f["area"] = perimeter, area
    f["compactness"] = 4 * np.pi * area / (perimeter ** 2 + 1e-6)
    return f


RGB, DIFFS = ["red", "blue", "green"], ["diff_r_g", "diff_r_b", "diff_b_g"]
BASE_11 = RGB + DIFFS + ["hue", "saturation", "value", "gradient", "brightness"]
NORM = ["norm_r", "norm_g", "norm_b"]

# name: (features, severity printed by the original notebook for scan 1, group counted as lesion in that trial)
FEATURE_SETS = {
    "original_6":              (RGB + DIFFS, 6.87, 1),
    "hsv_11":                  (RGB + ["hue", "saturation", "value"] + DIFFS + ["sat_norm", "val_norm"], 7.55, 1),
    "hsv_only_5":              (["hue", "saturation", "value", "sat_norm", "val_norm"], 74.97, 1),
    "texture_morphology_8":    (["hue", "saturation", "value", "gradient", "gabor", "perimeter", "area", "compactness"], 55.89, 1),
    "ai_13":                   (["row", "col"] + RGB + DIFFS + ["hue", "saturation", "value", "gradient", "brightness"], None, 2),
    "ai_no_gradient_12":       (["row", "col"] + RGB + DIFFS + ["hue", "saturation", "value", "brightness"], None, 2),
    "ai_no_brightness_12":     (["row", "col"] + RGB + DIFFS + ["hue", "saturation", "value", "gradient"], None, 2),
    "ai_no_hsv_10":            (["row", "col"] + RGB + DIFFS + ["gradient", "brightness"], None, 2),
    "ai_no_differences_10":    (["row", "col"] + RGB + ["hue", "saturation", "value", "gradient", "brightness"], None, 2),
    "ai_rgb_diffs_hsv_11":     (["row", "col"] + RGB + DIFFS + ["hue", "saturation", "value"], None, 2),
    "base_11":                 (BASE_11, 7.28, 1),
    "base_11_+_all_extras_18": (BASE_11 + ["rel_row", "rel_col", "dist_to_center"] + NORM + ["lbp"], 7.55, 1),
    "base_11_+_pos_lbp_15":    (BASE_11 + ["rel_row", "rel_col", "dist_to_center", "lbp"], 7.93, 1),
    "base_11_+_norm_14":       (BASE_11 + NORM, 57.59, 1),
    "base_11_+_dist_norm_lbp_16": (BASE_11 + ["dist_to_center"] + NORM + ["lbp"], 7.52, 1),
    "base_11_+_dist_norm_15":  (BASE_11 + ["dist_to_center"] + NORM, 57.59, 1),
    "no_hsv_13":               (RGB + DIFFS + ["gradient", "brightness", "dist_to_center"] + NORM + ["lbp"], 57.14, 1),
    "final_12":                (RGB + DIFFS + ["gradient", "dist_to_center"] + NORM + ["lbp"], 7.85, 2),
}


def order_clusters(labels, centers):
    """Renumber the groups by increasing sum of their centre coordinates (as in the original code)."""
    order = np.argsort(centers.sum(axis=1))
    remap = {old: new for new, old in enumerate(order)}
    return np.vectorize(remap.get)(labels)


def segment(path, features, random_state=2206):
    """K-means (3 groups, standardised features). Returns the ordered label image and the RGB image."""
    img = read_rgb(path)
    table = build_features(img)
    X = preprocessing.StandardScaler().fit_transform(table[features].to_numpy(dtype=float))
    km = KMeans(n_clusters=3, random_state=random_state, n_init=10).fit(X)
    return order_clusters(km.labels_, km.cluster_centers_).reshape(img.shape[:2]), img


def describe_clusters(img_rgb, ordered):
    """Mean raw RGB colour and share of pixels of each ordered group."""
    flat, lab = img_rgb.reshape(-1, 3), ordered.ravel()
    return pd.DataFrame({g: {"red": flat[lab == g, 0].mean(), "green": flat[lab == g, 1].mean(),
                             "blue": flat[lab == g, 2].mean(), "share_%": 100 * np.mean(lab == g)}
                         for g in np.unique(lab)}).T.round(1)


def severity_from_groups(ordered, lesion_label=2, healthy_label=0):
    lesion, healthy = np.sum(ordered == lesion_label), np.sum(ordered == healthy_label)
    return 100 * lesion / (lesion + healthy)

In [ ]:
# Look at every trial on scan 1 (as I did), and print the groups' mean colour to see what each group is.
results = {}
for name, (features, original, lesion_label) in FEATURE_SETS.items():
    ordered, img = segment(SCAN_1, features)
    results[name] = ordered
    print(f"== {name} ({len(features)} features)  | group counted as lesion: {lesion_label}"
          f"  | severity printed in the original notebook: {original}"
          f"  | recomputed here: {severity_from_groups(ordered, lesion_label):.2f}%")
    print(describe_clusters(img, ordered))

fig, axes = plt.subplots(3, 6, figsize=(24, 12))
for ax, (name, ordered) in zip(axes.ravel(), results.items()):
    ax.imshow(ordered, cmap="Oranges_r"); ax.set_title(name, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. A warning about the group numbers

The severity is `lesion / (lesion + healthy)` using the ordered group numbers, and **the group counted as lesion was
not always the same** in the original notebook (it is the third element of each entry of `FEATURE_SETS`):

* group **1** was the lesion (and group 0 the healthy tissue) in `original_6`, `hsv_11`, `hsv_only_5`,
  `texture_morphology_8` and in every trial from `base_11` to `no_hsv_13`;
* group **2** was the lesion in the six AI-suggested trials (`ai_*`) and in `final_12`.

The convention was changed by hand from one block of cells to the next, and I did not write down why. So the severities
printed by the original notebook for scan 1 do not all mean the same thing: most trials give about 7–8 %, three give
about 57 % and two earlier ones 56 % and 75 %. The notebook does not record why, and I did not check whether those
jumps were wrongly numbered groups or real segmentation failures. The loop above prints, next to the original value, the
severity recomputed with each trial's own convention, and the mean colour of each group so that you can check the
numbering by eye.

The 30/04/2025 progress report had already found the underlying problem with the 6-feature model: in 2 of the 6 scans the
numbering of the groups came out different (the lesion was group 2 and the background group 1), so the percentage was
computed against the background (56.5 % and 58.4 %, where the other four scans gave between 5.5 % and 6.9 %). The
proposed fix (writing the group numbers by hand after looking at the picture) was not implemented. The final function
keeps the same ordering rule and hard-codes group 2 as lesion; the ROI mask of the final tool (only pixels inside the
detected leaflets) limits how much a wrongly numbered background group could matter, but the ordering is still never
checked.

`check_group_numbers` prints, for each of the six scans, the mean colour of the three groups with the final features.

In [ ]:
def check_group_numbers(scans=range(1, 7), features=FEATURE_SETS["final_12"][0]):
    for n in scans:
        ordered, img = segment(SCANS_DIR / f"IMG_20250424_000{n}.tif", features)
        print(f"scan {n}: severity with group 2 = lesion, group 0 = healthy: {severity_from_groups(ordered):.2f}%")
        print(describe_clusters(img, ordered), "\n")


check_group_numbers()

## 3. Counting leaflets

The counter finds the leaflets as blobs: grey image, Gaussian blur, Otsu threshold (inverted), morphological
opening/closing, contours, then filters by area and by brightness. The expected number is 4 leaflets per node; the
difference is the missing leaflets, and `defoliation = missing / expected`.

Versions in the order I ran them (first test: **3 of the 6 scans counted 2 leaflets too many**, because two black
borders in the upper and lower right corners of the scans, artefacts of the scanner, were counted as leaflets):

| Version | Blur | Morphology | Filters |
|---|---|---|---|
| `v1` | Gaussian 7 × 7 | ellipse 5 × 5, 2 iterations open and close | area ≥ 100 |
| `v2` | same | same | + discard contours touching the image border |
| `v3` | Gaussian 3 × 3 | ellipse 5 × 5, 1 iteration | area ≥ 100, + discard dark blobs (mean grey < 100) |
| `v4` | Gaussian 5 × 5 | ellipse 5 × 5, 1 iteration | area ≥ 800, optional maximum area, dark threshold 100 |
| `v5` | Gaussian 5 × 5 | ellipse 3 × 3, 1 iteration | same as `v4` |
| **final** | **Gaussian 3 × 3, SD 1.5** | **ellipse 3 × 3, 1 iteration open + 1 close** | area ≥ 800, dark threshold 100 |

Then two systematic searches over the six scans (ten counts each), in May 2025 (progress report of 21/05/25). The
real number of leaflets of each scan was counted by hand.

In [ ]:
def count_leaflets(path, blur=(3, 3), sigma=1.5, element=3, ops=(("open", 1), ("close", 1)),
                   min_area=800, max_area=None, dark_threshold=100, drop_border_touching=False, return_image=False):
    """Leaflet counter. The defaults are the final parameters (the ones used by the package)."""
    bgr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)[:, :, :3]
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    if blur:
        gray = cv2.GaussianBlur(gray, blur, sigma)
    mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (element, element))
    operations = {"open": lambda m, n: cv2.morphologyEx(m, cv2.MORPH_OPEN, kernel, iterations=n),
                  "close": lambda m, n: cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel, iterations=n),
                  "dilate": lambda m, n: cv2.dilate(m, kernel, iterations=n),
                  "erode": lambda m, n: cv2.erode(m, kernel, iterations=n)}
    for name, iterations in ops:
        mask = operations[name](mask, iterations)

    height, width = mask.shape
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    leaflets = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < min_area or (max_area is not None and area > max_area):
            continue
        if drop_border_touching:
            x, y, w, h = cv2.boundingRect(c)
            if x <= 1 or y <= 1 or x + w >= width - 1 or y + h >= height - 1:
                continue
        if dark_threshold:
            blob = np.zeros_like(mask)
            cv2.drawContours(blob, [c], -1, 255, thickness=cv2.FILLED)
            if cv2.mean(gray, mask=blob)[0] < dark_threshold:
                continue
        leaflets.append(c)
    if return_image:
        out = bgr.copy(); cv2.drawContours(out, leaflets, -1, (0, 255, 0), 2)
        return len(leaflets), cv2.cvtColor(out, cv2.COLOR_BGR2RGB)
    return len(leaflets)


COUNTER_VERSIONS = {
    "v1":    dict(blur=(7, 7), sigma=0, element=5, ops=(("open", 2), ("close", 2)), min_area=100, dark_threshold=0),
    "v2":    dict(blur=(7, 7), sigma=0, element=5, ops=(("open", 2), ("close", 2)), min_area=100, dark_threshold=0,
                  drop_border_touching=True),
    "v3":    dict(blur=(3, 3), sigma=0, element=5, ops=(("open", 1), ("close", 1)), min_area=100, dark_threshold=100),
    "v4":    dict(blur=(5, 5), sigma=0, element=5, ops=(("open", 1), ("close", 1)), min_area=800, dark_threshold=100),
    "v5":    dict(blur=(5, 5), sigma=0, element=3, ops=(("open", 1), ("close", 1)), min_area=800, dark_threshold=100),
    "final": dict(),
}


def run_sweep(configs, scans=range(1, 7)):
    """Count the leaflets of every scan under every configuration (needs the scans)."""
    return pd.DataFrame({name: {f"IMG_20250424_000{n}.tif": count_leaflets(SCANS_DIR / f"IMG_20250424_000{n}.tif", **kw)
                                for n in scans} for name, kw in configs.items()})


# Uncomment to re-run the counter versions on the scans:
# print(run_sweep(COUNTER_VERSIONS))

### Gaussian blur and morphology sweeps (real results, no scans needed)

The two tables below are the results I saved in 2025 (`data/leaflet_counts_gaussian_sweep.csv` and
`data/leaflet_counts_morphology_sweep.csv`). Each column is one configuration; the table of errors is
`counted − real`.

* Gaussian sweep: kernels 3 × 3, 5 × 5 and 7 × 7 without SD, with SD 1.5 and with SD 2, and without blur.
* Morphology sweep (with the best blur, 3 × 3 and SD 1.5): the kernel and the iterations of open and close,
  and variants with only open, only close, or an extra dilate or erode.

In [ ]:
gaussian = pd.read_csv(DATA_DIR / "leaflet_counts_gaussian_sweep.csv").set_index("scan")
morphology = pd.read_csv(DATA_DIR / "leaflet_counts_morphology_sweep.csv").set_index("scan")


def errors(table):
    real = table["real_leaflets"]
    return table.drop(columns="real_leaflets").sub(real, axis=0)


for title, table in [("Gaussian blur sweep", gaussian), ("Morphology sweep", morphology)]:
    err = errors(table)
    print(f"== {title}: counted - real, per scan")
    print(err.T.assign(total_abs_error=err.abs().sum().values, exact_scans=(err == 0).sum().values)
             .sort_values(["total_abs_error"]))
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (title, table) in zip(axes, [("Gaussian blur sweep", gaussian), ("Morphology sweep", morphology)]):
    x = np.arange(len(table))
    for col in table.columns.drop("real_leaflets"):
        ax.plot(x, table[col], "o", alpha=0.5, label=col)
    ax.plot(x, table["real_leaflets"], "k_", markersize=22, markeredgewidth=3, label="real")
    ax.set_xticks(x, [f"scan {i + 1}" for i in x]); ax.set_ylabel("Leaflets"); ax.set_title(title)
    ax.legend(fontsize=6, ncol=2)
plt.tight_layout(); plt.show()

**What the sweeps say.** Counts equal to the real number in 5 of the 6 scans, and one scan always off by one
(38 counted, 39 real, scan 1), were obtained by Gaussian 3 × 3 with SD 1.5 (and also by 3 × 3 with SD 2) combined with
ellipse 3 × 3 and one iteration of open and one of close. Those are the parameters of the final tool. The same counts
came out of five other morphology variants (only open once, only close once, open twice, and open + close with an
extra dilate or an extra erode); the progress report of 21/05/25 lists four of them and does not mention the
"only close" one, which gives the same counts in the saved table.

With six scans, "best" means "fewest counting errors on the six scans I used to choose": there is no held-out set.

## 4. Defoliation: how to combine it with the severity

Once the leaflets are counted, the missing ones give `defoliation = missing / expected` (expected = 4 per node;
the scans had 10, 10, 9, 11, 12 and 12 nodes). I tried five ways of combining it with the lesion severity (in the
notebook of June 2025) and kept the multiplicative one:

| Combination | Formula |
|---|---|
| simple average | (severity + defoliation) / 2 |
| weighted | 0.7 × severity + 0.3 × defoliation |
| maximum | max(severity, defoliation) |
| capped sum | min(1, severity + defoliation) |
| **multiplicative (kept)** | **1 − (1 − severity) × (1 − defoliation)** |

The multiplicative form reads as "the proportion of the expected leaf area that is lesion or missing", and cannot
go above 100 %. The notebook does not record why it was preferred over the others.

In [ ]:
NODES = {1: 10, 2: 10, 3: 9, 4: 11, 5: 12, 6: 12}   # nodes per scan, as entered by hand in the original run


def defoliation_variants(severity, defoliation):
    """The five ways of combining severity and defoliation that I compared (all fractions in 0-1)."""
    return {"average": (severity + defoliation) / 2,
            "weighted_0.7_0.3": 0.7 * severity + 0.3 * defoliation,
            "maximum": max(severity, defoliation),
            "capped_sum": min(1, severity + defoliation),
            "multiplicative": 1 - (1 - severity) * (1 - defoliation)}


# Usage: defoliation_variants(severity, defoliation), both as fractions between 0 and 1

## 5. Putting it together

The final tool does the whole thing, `lls_sev_est/lls_sev_est.py` (see the main README for how to use it): detect
leaflets, count them and compute defoliation, protect the edges, segment with the 12 features, compute severity only
inside the protected leaflets, and the adjusted (multiplicative) severity. The counter parameters and the 12 features
in the package are the ones in this notebook.

**Edge protection** (shaving 7 px off each leaflet so that the shadow on the border is not counted as lesion) appears
only in the package: no notebook or report documents how that value was chosen.

In [ ]:
# Optional: run the packaged tool on scan 1 (10 nodes -> 40 leaflets expected, 39 present)
# from lls_sev_est import lls_sev_est
# results = pd.DataFrame(columns=["image_route", "n_leaflets", "nodes", "missing_leaflets",
#                                 "severity_%", "defoliation_%", "adjusted_severity_%"])
# results = lls_sev_est(str(SCAN_1), nodes=NODES[1], results_df=results, show_results=True)
# results